# 🧠 ความแม่นยำ (Precision) และการกรองเกณฑ์ความมั่นใจของกล่องวัตถุ

ยินดีต้อนรับสู่โน้ตบุ๊กประกอบการอธิบายเรื่อง **ความแม่นยำ (Precision)**! ในโน้ตบุ๊กนี้เราจะ:
1. ทำความรู้จักนิยามสูตรคณิตศาสตร์พื้นฐานของค่า Precision
2. เขียนฟังก์ชันคำนวณ Precision จากศูนย์และทดสอบเทียบความถูกต้องกับ `scikit-learn`
3. จำลองชุดข้อมูลตรวจจับหัวบ่อน้ำมันจากโดรน เพื่อศึกษาพฤติกรรมของผลทำนายบวก (Positive Predictions) ภายใต้เกณฑ์ระดับความมั่นใจ (Confidence Thresholds) ต่างๆ
4. พล็อตกราฟ **โค้งความแม่นยำต่อเกณฑ์ความมั่นใจ (Precision-Threshold Curve)** เพื่อสังเกตวิธีกรองค่าลวงบวก (False Positives หรือสัญญาณเตือนภัยเท็จ) ด้วยการปรับระดับความมั่นใจ
5. อธิบายการตั้งค่าเกณฑ์ความมั่นใจใน YOLO ที่ส่งผลต่อต้นทุนการดำเนินงานจริงในภาคอุตสาหกรรม

เริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อนครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import precision_score

# กำหนดค่า seed เพื่อให้ได้ผลลัพธ์การสุ่มเหมือนเดิมทุกครั้ง
np.random.seed(42)

## 1. การคำนวณค่า Precision จากพื้นฐาน (Calculating Precision from Scratch)

เราจะพัฒนาฟังก์ชันเพื่อคำนวณสัดส่วนค่า Precision ตามนิยามเชิงสูตรคณิตศาสตร์:
$$\text{Precision} = \frac{TP}{TP + FP}$$

In [ ]:
def calculate_precision(y_true, y_pred):
    """
    คำนวณค่า Precision จากศูนย์
    """
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    
    total_pred_positive = TP + FP
    if total_pred_positive == 0:
        return 1.0
    return TP / total_pred_positive

# กำหนดอาร์เรย์ตัวอย่างจำลองสำหรับการเปรียบเทียบ (เช่น การตรวจสอบคลาสของ Bounding Box)
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 1, 0, 0, 1, 1, 0, 1, 0])

prec_scratch = calculate_precision(y_true, y_pred)
prec_sklearn = precision_score(y_true, y_pred)

print(f"Custom Precision : {prec_scratch:.4f}")
print(f"Sklearn Precision: {prec_sklearn:.4f}")

## 2. การวิเคราะห์ความสัมพันธ์ระหว่างระดับความมั่นใจของกล่องวัตถุและค่า Precision

ในระบบตรวจจับวัตถุ ทุกกรอบสี่เหลี่ยม Bounding Box ที่ทำนายออกมาจะมีค่าคะแนนความน่าจะเป็นแนบมาด้วย เรียกว่าค่า **ระดับความมั่นใจ (Confidence)**
เราจะใช้ **เกณฑ์ระดับความมั่นใจ (Confidence Threshold)** มาเป็นตัวกรองว่าจะเก็บกล่องข้อความใดไว้หรือปัดทิ้งเป็นฉากหลัง:
-   หากค่าความมั่นใจของกล่องมีค่า $\ge \text{threshold}$ จะเก็บไว้เป็นคลาส 1 (ทายว่าใช่)
-   หากค่าความมั่นใจมีค่าน้อยกว่า $< \text{threshold}$ จะถือว่าเป็นคลาส 0 (ทิ้งเป็นฉากหลัง Background)

เราจะลองสุ่มสร้างกล่อง Bounding Box ขึ้นมา 200 กล่องที่มีการระบุเฉลยจริง (เป็นหัวบ่อน้ำมัน หรือเป็นฉากหลัง) ร่วมกับคะแนนความมั่นใจทำนายจากโมเดล เพื่อนำมาวิเคราะห์ว่าการปรับเกณฑ์ความมั่นใจส่งผลโดยตรงต่อค่า Precision อย่างไรบ้าง

In [ ]:
# สร้างข้อมูลจำลอง 200 ตัวอย่าง
n_samples = 200

# หัวบ่อแก๊สจริง 30 ตัวอย่าง (คลาส 1) และฉากหลัง/สัญญาณรบกวน 170 ตัวอย่าง (คลาส 0)
y_true_wellheads = np.concatenate([np.ones(30), np.zeros(170)]).astype(int)

# สร้างคะแนนความมั่นใจจำลอง:
# กลุ่มหัวบ่อน้ำมันจริงกำหนดให้มีความมั่นใจเฉลี่ยสูง (mean 0.8)
conf_wellheads = np.random.normal(0.8, 0.15, 30)
# กลุ่มฉากหลังทั่วไปกำหนดให้มีความมั่นใจเฉลี่ยต่ำ (mean 0.3)
conf_bg = np.random.normal(0.3, 0.18, 170)

# รวมคะแนนความมั่นใจเข้าด้วยกันและจำกัดช่วงให้อยู่ระหว่าง 0.0 ถึง 1.0
confidence_scores = np.concatenate([conf_wellheads, conf_bg])
confidence_scores = np.clip(confidence_scores, 0.0, 1.0)

จากนั้น เราจะทำการรันหาค่า Precision ไล่เรียงระดับเกณฑ์ความมั่นใจตั้งแต่ค่า `0.0` ไปจนถึงสูงสุดที่ `0.95`

In [ ]:
thresholds = np.linspace(0.0, 0.95, 100)
precision_history = []
detections_kept = []

for threshold in thresholds:
    y_pred_temp = (confidence_scores >= threshold).astype(int)
    prec = calculate_precision(y_true_wellheads, y_pred_temp)
    precision_history.append(prec)
    detections_kept.append(np.sum(y_pred_temp == 1))

# พล็อตกราฟเส้นความแม่นยำเทียบกับเกณฑ์ความมั่นใจ (Precision-Threshold Curve)
plt.figure(figsize=(10, 5))
plt.plot(thresholds, precision_history, color='teal', linewidth=3, label='Precision')
plt.xlabel('Confidence Threshold')
plt.ylabel('Precision Score')
plt.title('Precision vs. Confidence Threshold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.axvline(0.5, color='red', linestyle='--', alpha=0.7, label='ค่าเกณฑ์เริ่มต้นมาตรฐานทั่วไป (0.50)')
plt.legend()
plt.show()

สังเกตลักษณะกราฟดังนี้ครับ:
-   หากเราตั้งเกณฑ์ความมั่นใจไว้ต่ำมาก (เช่น `0.2`) ค่า Precision จะดิ่งลงต่ำมากเหลือเพียงแค่ประมาณ ($\approx 25\%$) ซึ่งแปลว่าสัญญาณแจ้งเตือนภัยที่ส่งไปหาผู้ปฏิบัติงานจริง จะกลายเป็น **การแจ้งเตือนเตือนภัยเท็จ (False Alarms) ถึง 75%**!
-   หากเราเขยิบเกณฑ์ไปไว้สูงๆ (เช่น `0.75`) ค่า Precision จะกระโดดสูงขึ้นจนถึง **$100\%$** ซึ่งหมายความว่า ทุกสัญญาณการแจ้งเตือนบ่อน้ำมันรั่วไหลที่ระบบตรวจเจอจะเป็นของจริงทั้งหมด ช่วยตัดปัญหาการส่งช่างซ่อมบำรุงออกไปหน้างานเสียเปล่าจากสัญญาณเท็จ

มาดูผลตอบแทนหรือความสูญเสียในมุมกลับกันบ้างครับ: จากเป้าหมายทั้งหมดเราตามหาหัวบ่อน้ำมันจริงพบกี่จุด?

In [ ]:
# พล็อตกราฟแสดงปริมาณกล่อง Bounding Box ที่ถูกเก็บไว้หลังผ่านการกรอง
plt.figure(figsize=(10, 5))
plt.plot(thresholds, detections_kept, color='purple', linewidth=2.5, label='Number of Bounding Boxes Kept')
plt.xlabel('Confidence Threshold')
plt.ylabel('Count')
plt.title('Total Positive Detections Kept vs. Threshold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

เมื่อค่าเกณฑ์ถูกขยับสูงขึ้น ปริมาณกล่อง Bounding Box ที่ระบบยอมแสดงผลลัพธ์ว่าใช่จะลดจำนวนลงอย่างรวดเร็ว ในระดับเกณฑ์ที่สูงมาก เราอาจทำหัวบ่อน้ำมันของจริงตกหล่นหายไปบ้าง (ทำให้ค่า Recall ต่ำลง) ซึ่งพฤติกรรมนี้สะท้อนกรอบคิดเรื่อง **การเลือกระหว่างความแม่นยำและความครอบคลุม (Precision-Recall Tradeoff)**

## 💡 ความเชื่อมโยงสู่ Computer Vision และ YOLO
*   **การกรองผ่านเกณฑ์ระดับความมั่นใจ (อาร์กิวเมนต์ `conf`):** เมื่อเรารันทำนายผลด้วยโมเดล YOLO เราสามารถกำหนดเป้าหมายค่าความแม่นยำได้เองผ่านการควบคุมตัวแปรพารามิเตอร์ `conf`:
    ```bash
    yolo predict model=yolo26n.pt source=drone.mp4 conf=0.75
    ```
    หากเป้าหมายหลักในภาคอุตสาหกรรมคือการตัดปัญหาความสูญเสียจากสัญญาณเตือนลวง (เช่น การสแกนภาพหัวบ่อน้ำมันที่อยู่ไกลมาก) การตั้งค่า `conf=0.75` จะบีบให้ตัวตรวจจับรายงานเฉพาะกล่องที่มีระดับความแม่นยำสูงพิเศษเท่านั้น